# Real-event inference sanity check: M08 + Mondrian conformal prediction

This notebook explores whether the M08 CNN + Mondrian conformal prediction pipeline trained on simulated BBH signals can be applied to real GWOSC/LVK events as a first-order low-latency parameter-estimation method.

The goal is not to claim formal conformal validity on real detector data. The conformal calibration was performed on simulated signals, so real-event inference is affected by domain shift: real non-stationary detector noise, glitches, calibration uncertainty, PSD mismatch, detector availability, and waveform-systematics differences.

The notebook proceeds incrementally:

1. Load the exact training/generation configuration.
2. Reconstruct the M08 model.
3. Load label normalization statistics.
4. Validate inference on controlled inputs.
5. Reconstruct/apply Mondrian calibrators.
6. Build real GWOSC event inputs.
7. Compare predictions and intervals with published LVK values.

Construir el input real para uno o pocos eventos HLV, pasar M08, aplicar el calibrador Mondrian y comparar cualitativamente con LVK. No estimar cobertura real todavía.

## 1. Imports and Paths

In [56]:
from pathlib import Path
import json
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Ajusta estas rutas a tu entorno actual

print("Current working directory:", os.getcwd())

#PROJECT_ROOT = Path("/home/victor/gw/cbc_pe")
#DATA_ROOT = Path("/home/victor/gw/cbc_pe/data")
PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe")
DATA_ROOT = Path("/data/vserrano/cbc_pe_data")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT does not exist: {PROJECT_ROOT}")
else:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

CONFIG_DIR = PROJECT_ROOT / "configs" 
MODEL_DIR = DATA_ROOT / "models" / "checkpoints" / DATASET_ID
RESULTS_DIR = DATA_ROOT / "results" / DATASET_ID

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CONFIG_DIR exists:", CONFIG_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

Current working directory: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks
PROJECT_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
DATA_ROOT: /data/vserrano/cbc_pe_data
CONFIG_DIR exists: True
MODEL_DIR exists: True
RESULTS_DIR exists: True


## 2. Load configs and parameters

In [57]:
generation_config_path =  PROJECT_ROOT / "configs" / "generation" / "generate_500k_bbh_4s.json"
training_config_path = PROJECT_ROOT / "configs" / "experiments" / "train_500k_M08_resdilated_emb64_d124_bs256_seed123.json"

with open(generation_config_path, "r") as f:
    gen_cfg = json.load(f)

with open(training_config_path, "r") as f:
    train_cfg = json.load(f)

print("Generation config keys:", gen_cfg.keys())
print("Training config keys:", train_cfg.keys())
print("Training parameters:", train_cfg["model"]["kwargs"].keys())

Generation config keys: dict_keys(['project_root', 'data_root', 'output', 'generation', 'simulation', 'parameter_sampler', 'detectors', 'signal_processor', 'label_transformer'])
Training config keys: dict_keys(['project_root', 'data_root', 'dataset', 'model', 'training', 'outputs'])
Training parameters: dict_keys(['n_detectors', 'n_outputs', 'embedding_dim', 'residual_channels', 'dilations', 'residual_kernel_size', 'dropout_conv', 'dropout_dense', 'num_groups'])


## 3. Signal requirements / conditions

In [58]:
detectors = gen_cfg["detectors"]

fs = 4096
duration = gen_cfg["simulation"]["duration"]
n_samples = int(duration * fs)

context_start = gen_cfg["simulation"]["processing_context_start_samples"]
context_end = gen_cfg["simulation"]["processing_context_end_samples"]
processing_length = n_samples + context_start + context_end

signal_processor_cfg = gen_cfg["signal_processor"]

print("Detector order:", detectors)
print("Sampling frequency:", fs)
print("Final duration:", duration)
print("Final samples:", n_samples)
print("Processing context start samples:", context_start)
print("Processing context end samples:", context_end)
print("Processing input length:", processing_length)
print("Processing input duration:", processing_length / fs)

print("\nSignal processor:")
for k, v in signal_processor_cfg.items():
    print(f"  {k}: {v}")

Detector order: ['H1', 'L1', 'V1']
Sampling frequency: 4096
Final duration: 4.0
Final samples: 16384
Processing context start samples: 1664
Processing context end samples: 1664
Processing input length: 19712
Processing input duration: 4.8125

Signal processor:
  whitening_method: psd
  apply_highpass: True
  apply_lowpass: True
  apply_standardization: False
  output_mode: crop_to_config
  whitening_low_frequency_cutoff: 30.0
  whitening_max_filter_duration: 0.5
  whitening_trunc_method: hann
  highpass_frequency: 30.0
  lowpass_frequency: 512.0
  fir_order: 256
  fir_beta: 5.0
  remove_corrupted: True


In [59]:
### SANITY CHECKS to assure that the configuration parameters are consistent with the expected values

assert detectors == ["H1", "L1", "V1"], detectors
assert fs == 4096
assert n_samples == 16384
assert processing_length == 19712

assert signal_processor_cfg["whitening_method"] == "psd"
assert signal_processor_cfg["apply_highpass"] is True
assert signal_processor_cfg["apply_lowpass"] is True
assert signal_processor_cfg["apply_standardization"] is False
assert signal_processor_cfg["highpass_frequency"] == 30.0
assert signal_processor_cfg["lowpass_frequency"] == 512.0

print("Input contract validated.")

Input contract validated.


## 4. Imports (model, simulation-config)

In [60]:
from src.models.network import SimpleCNN_ResidualDilated

# Getting model info
model_cfg = train_cfg["model"]

print(model_cfg["class_name"])
print(model_cfg["kwargs"])

model = SimpleCNN_ResidualDilated(**model_cfg["kwargs"])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

SimpleCNN_ResidualDilated
{'n_detectors': 3, 'n_outputs': 3, 'embedding_dim': 64, 'residual_channels': 64, 'dilations': [1, 2, 4], 'residual_kernel_size': 7, 'dropout_conv': 0.05, 'dropout_dense': 0.1, 'num_groups': 8}
SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=Fals

In [61]:
### Shape test

x_dummy = torch.zeros((2, 3, 16384), dtype=torch.float32)

with torch.no_grad():
    y_dummy, emb_dummy = model(x_dummy, return_embedding=True)

print("y_dummy shape:", y_dummy.shape)
print("emb_dummy shape:", emb_dummy.shape)

y_dummy shape: torch.Size([2, 3])
emb_dummy shape: torch.Size([2, 64])


## 5. Load the checkpoint

In [62]:
!ls  /home/victor/gw/cbc_pe/data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000

ls: cannot access '/home/victor/gw/cbc_pe/data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000': No such file or directory


In [63]:
print (MODEL_DIR)

/data/vserrano/cbc_pe_data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000


In [64]:
checkpoint_tag = train_cfg["outputs"]["checkpoint_tag"]
print("Checkpoint tag:", checkpoint_tag)

candidate_checkpoints = sorted(MODEL_DIR.rglob(f"*{checkpoint_tag}*"))
for p in candidate_checkpoints[:20]:
    print(p)

print("Number of candidates:", len(candidate_checkpoints))

Checkpoint tag: M08_resdilated_emb64_d124
/data/vserrano/cbc_pe_data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_MSELoss_seed124_checkpoint.pt
Number of candidates: 1


In [65]:
checkpoint_path = candidate_checkpoints[-1]  # Load the last checkpoint

ckpt = torch.load(checkpoint_path, map_location="cpu")
print(ckpt.keys())

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'best_val_loss', 'y_mean', 'y_std', 'model_config', 'training_config', 'elapsed_seconds', 'history'])


SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (residual_blocks): Sequential(
    (0): ResidualDilatedBlock(
      (conv1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
      (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
      (activation

In [66]:
with torch.no_grad():
    y_dummy, emb_dummy = model(x_dummy, return_embedding=True)

print(y_dummy)
print(emb_dummy.shape)

tensor([[-2.0899, -2.6385,  0.7115],
        [-2.0899, -2.6385,  0.7115]])
torch.Size([2, 64])


## 6. Label normalization statistics

In [67]:
# ------------------------------------------------------------
# 6. Label normalization statistics
# ------------------------------------------------------------

y_mean = np.asarray(ckpt["y_mean"], dtype=np.float64)
y_std = np.asarray(ckpt["y_std"], dtype=np.float64)

label_names = ["chirp_mass", "total_mass", "chi_eff"]

print("Label names:", label_names)
print("y_mean:", y_mean)
print("y_std:", y_std)

assert y_mean.shape == (3,)
assert y_std.shape == (3,)
assert np.all(np.isfinite(y_mean))
assert np.all(np.isfinite(y_std))
assert np.all(y_std > 0)

print("Label statistics validated.")

Label names: ['chirp_mass', 'total_mass', 'chi_eff']
y_mean: [3.74526215e+01 9.49739685e+01 1.02269521e-03]
y_std: [16.48472214 34.65016556  0.44085518]
Label statistics validated.


In [68]:
def inverse_standardize(y_std_space):
    """
    Convert standardized labels/predictions to physical units:
    [chirp_mass, total_mass, chi_eff].
    """
    y_std_space = np.asarray(y_std_space, dtype=np.float64)
    return y_std_space * y_std + y_mean


def standardize(y_phys):
    """
    Convert physical labels to standardized training space.
    """
    y_phys = np.asarray(y_phys, dtype=np.float64)
    return (y_phys - y_mean) / y_std

test_std = np.zeros((1, 3))
test_phys = inverse_standardize(test_std)

print("Zero standardized corresponds to physical mean:")
for name, value in zip(label_names, test_phys[0]):
    print(f"{name}: {value:.6g}")

Zero standardized corresponds to physical mean:
chirp_mass: 37.4526
total_mass: 94.974
chi_eff: 0.0010227


## 7. Función de inferencia

Hace las predicciones usando el modelo entrenado 

In [69]:
def predict_m08(model, X, device="cpu"):
    """
    Run M08 inference.

    Parameters
    ----------
    model : torch.nn.Module
        Loaded M08 model.
    X : np.ndarray
        Shape (n_events, 3, 16384) or (3, 16384).

    Returns
    -------
    pred_std : np.ndarray
        Standardized predictions, shape (n_events, 3).
    pred_phys : np.ndarray
        Physical predictions, shape (n_events, 3).
    emb : np.ndarray
        Embeddings, shape (n_events, 64).
    """
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 2:
        X = X[None, :, :]

    if X.shape[1:] != (3, 16384):
        raise ValueError(f"Expected X shape (N, 3, 16384), got {X.shape}")

    model = model.to(device)
    model.eval()

    x_tensor = torch.from_numpy(X).to(device)

    with torch.no_grad():
        pred_std_t, emb_t = model(x_tensor, return_embedding=True)

    pred_std = pred_std_t.cpu().numpy()
    emb = emb_t.cpu().numpy()
    pred_phys = inverse_standardize(pred_std)

    return pred_std, pred_phys, emb

In [70]:
X_dummy = np.zeros((1, 3, 16384), dtype=np.float32)

pred_std_dummy, pred_phys_dummy, emb_dummy = predict_m08(model, X_dummy)

print("pred_std_dummy:", pred_std_dummy)
print("pred_phys_dummy:", pred_phys_dummy)
print("emb_dummy shape:", emb_dummy.shape)

pred_std_dummy: [[-2.0899227  -2.6385005   0.71149665]]
pred_phys_dummy: [[3.00082701 3.54949102 0.31468968]]
emb_dummy shape: (1, 64)


## 8. Load embeddings and preds

In [71]:
# ------------------------------------------------------------
# Load M08 prediction/embedding file
# ------------------------------------------------------------

DATASET_ID = train_cfg["dataset"]["dataset_id"]

prediction_candidates = sorted(
    RESULTS_DIR.rglob(
        f"{DATASET_ID}_SimpleCNN_ResidualDilated_M08*predictions_embeddings*.npz"
    )
)

print("Prediction/embedding candidates:")
for p in prediction_candidates:
    print(" ", p)

print("Number of candidates:", len(prediction_candidates))

assert len(prediction_candidates) >= 1, "No M08 prediction/embedding file found."

M08_PRED_PATH = prediction_candidates[0]
print("Selected:", M08_PRED_PATH)

Prediction/embedding candidates:
  /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_MSELoss_seed124_val_cal_test_predictions_embeddings.npz
  /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08a_residual_emb64_d111_MSELoss_seed123_val_cal_test_predictions_embeddings.npz
Number of candidates: 2
Selected: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_MSELoss_seed124_val_cal_test_predictions_embeddings.npz


In [72]:
m08_data = np.load(M08_PRED_PATH, allow_pickle=True)

print("Available keys:")
for key in sorted(m08_data.files):
    value = m08_data[key]
    print(f"{key:25s} shape={value.shape} dtype={value.dtype}")

Available keys:
available_splits          shape=(3,) dtype=<U4
checkpoint_file           shape=() dtype=<U174
dataset_path              shape=() dtype=<U86
emb_cal                   shape=(30000, 64) dtype=float32
emb_test                  shape=(30000, 64) dtype=float32
emb_val                   shape=(40000, 64) dtype=float32
idx_cal                   shape=(30000,) dtype=int64
idx_test                  shape=(30000,) dtype=int64
idx_val                   shape=(40000,) dtype=int64
label_names               shape=(3,) dtype=<U10
label_stats_path          shape=() dtype=<U158
model_config              shape=() dtype=object
pred_cal                  shape=(30000, 3) dtype=float32
pred_test                 shape=(30000, 3) dtype=float32
pred_val                  shape=(40000, 3) dtype=float32
split_path                shape=() dtype=<U142
y_cal                     shape=(30000, 3) dtype=float32
y_mean                    shape=(3,) dtype=float32
y_std                     shape=(3,) dtype

In [73]:
SPLITS = {}

for split in ["val", "cal", "test"]:
    required = [f"pred_{split}", f"y_{split}", f"emb_{split}"]

    missing = [key for key in required if key not in m08_data.files]
    if missing:
        print(f"Skipping split={split}, missing keys:", missing)
        continue

    SPLITS[split] = {
        "pred": np.asarray(m08_data[f"pred_{split}"], dtype=np.float64),
        "y": np.asarray(m08_data[f"y_{split}"], dtype=np.float64),
        "emb": np.asarray(m08_data[f"emb_{split}"], dtype=np.float64),
    }

    idx_key = f"idx_{split}"
    if idx_key in m08_data.files:
        SPLITS[split]["idx"] = np.asarray(m08_data[idx_key])

for split, data in SPLITS.items():
    print(f"\nSplit: {split}")
    for key, value in data.items():
        print(f"  {key:5s}: {value.shape}")


Split: val
  pred : (40000, 3)
  y    : (40000, 3)
  emb  : (40000, 64)
  idx  : (40000,)

Split: cal
  pred : (30000, 3)
  y    : (30000, 3)
  emb  : (30000, 64)
  idx  : (30000,)

Split: test
  pred : (30000, 3)
  y    : (30000, 3)
  emb  : (30000, 64)
  idx  : (30000,)


## 9. Load Mondrian selected configs

In [74]:
# ------------------------------------------------------------
# 9. Load selected Mondrian configurations
# ------------------------------------------------------------

MONDRIAN_DIR = RESULTS_DIR / "mondrian_M08_final_baseline"

print("MONDRIAN_DIR:", MONDRIAN_DIR)
print("Exists:", MONDRIAN_DIR.exists())

for p in sorted(MONDRIAN_DIR.glob("*")):
    print(p.name)

MONDRIAN_DIR: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/mondrian_M08_final_baseline
Exists: True
conditional_metrics.csv
figures_academic
global_conformal.csv
global_point_metrics.csv
grid_aggregate_summary.csv
mondrian_summary_all.csv
point_metrics.csv
selected_configurations.csv
selected_systems_report.txt
selected_systems_summary.csv
test_metadata.csv
top_candidates.csv


In [75]:
import pandas as pd

selected_config_path = MONDRIAN_DIR / "selected_configurations.csv"
selected_systems_path = MONDRIAN_DIR / "selected_systems_summary.csv"

selection_df = pd.read_csv(selected_config_path)
selected_systems_df = pd.read_csv(selected_systems_path)

display(selection_df)
display(selected_systems_df)

,taxonomy_mode,interval_mode,n_bins,label,label_index,global_coverage,global_miscoverage,global_undercoverage_pvalue,global_mean_width_std,global_median_width_std,...,global_2sigma_low,global_2sigma_high,global_3sigma_low,global_3sigma_high,global_within_2sigma,min_width_for_label,width_limit,relative_width_excess,selection_policy,final_policy
0,prediction,asymmetric,24,chirp_mass,0,0.902767,0.097233,0.946419,0.905068,0.947766,...,0.896536,0.903464,0.894804,0.905196,True,15.439307,15.748093,0.01194,conservative_zero_under_bins_2sigma,conservative
1,difficulty,symmetric,24,chirp_mass,0,0.901567,0.098433,0.819569,0.896450,0.870898,...,0.896536,0.903464,0.894804,0.905196,True,14.356512,14.643642,0.00000,efficient_local_validity_tolerant,efficient
2,prediction,asymmetric,16,total_mass,1,0.901067,0.098933,0.733653,0.846922,0.805739,...,0.896536,0.903464,0.894804,0.905196,True,27.919006,28.477386,0.00000,conservative_zero_under_bins_2sigma,conservative
3,prediction,asymmetric,32,total_mass,1,0.901500,0.098500,0.809228,0.850464,0.787492,...,0.896536,0.903464,0.894804,0.905196,True,27.286712,27.832446,0.00000,efficient_local_validity_tolerant,efficient
4,difficulty,symmetric,6,chi_eff,2,0.898400,0.101600,0.180210,1.345089,1.307212,...,0.896536,0.903464,0.894804,0.905196,True,0.576291,0.587817,0.00000,conservative_zero_under_bins_2sigma,conservative
5,difficulty,symmetric,12,chi_eff,2,0.899067,0.100933,0.297681,1.344295,1.273241,...,0.896536,0.903464,0.894804,0.905196,True,0.561315,0.572541,0.00000,efficient_local_validity_tolerant,efficient


,final_policy,label,taxonomy_mode,interval_mode,n_bins,coverage,median_width_phys,q90_width_phys,q95_width_phys,min_coverage_per_bin,n_bins_under_2sigma,max_undercoverage_gap,tail_miss_imbalance
0,conservative,chirp_mass,prediction,asymmetric,24,0.902767,15.623656,21.191378,21.730857,0.886381,0,0.013619,0.000233
1,conservative,total_mass,prediction,asymmetric,16,0.901067,27.919006,37.565449,38.805998,0.887652,0,0.012348,0.001933
2,conservative,chi_eff,difficulty,symmetric,6,0.898400,0.576291,0.812271,0.812271,0.892167,0,0.007833,0.002067
3,efficient,chirp_mass,difficulty,symmetric,24,0.901567,14.356512,23.366214,24.942036,0.873544,1,0.026456,0.002367
4,efficient,total_mass,prediction,asymmetric,32,0.901500,27.286712,37.805153,38.552800,0.874737,2,0.025263,0.001833
5,efficient,chi_eff,difficulty,symmetric,12,0.899067,0.561315,0.721185,0.894796,0.882232,1,0.017768,0.002133


In [76]:


MONDRIAN_POLICY = "conservative"

selected_policy_df = selected_systems_df[
    selected_systems_df["final_policy"] == MONDRIAN_POLICY
].copy()

display(selected_policy_df)

assert len(selected_policy_df) == 3
assert set(selected_policy_df["label"]) == set(label_names)

selected_configs = {}

for _, row in selected_policy_df.iterrows():
    label = row["label"]
    selected_configs[label] = {
        "label": label,
        "label_index": label_names.index(label),
        "taxonomy_mode": row["taxonomy_mode"],
        "interval_mode": row["interval_mode"],
        "n_bins": int(row["n_bins"]),
        "coverage": float(row["coverage"]),
        "median_width_phys": float(row["median_width_phys"]),
        "q90_width_phys": float(row["q90_width_phys"]),
        "q95_width_phys": float(row["q95_width_phys"]),
        "min_coverage_per_bin": float(row["min_coverage_per_bin"]),
    }

selected_configs

,final_policy,label,taxonomy_mode,interval_mode,n_bins,coverage,median_width_phys,q90_width_phys,q95_width_phys,min_coverage_per_bin,n_bins_under_2sigma,max_undercoverage_gap,tail_miss_imbalance
0,conservative,chirp_mass,prediction,asymmetric,24,0.902767,15.623656,21.191378,21.730857,0.886381,0,0.013619,0.000233
1,conservative,total_mass,prediction,asymmetric,16,0.901067,27.919006,37.565449,38.805998,0.887652,0,0.012348,0.001933
2,conservative,chi_eff,difficulty,symmetric,6,0.898400,0.576291,0.812271,0.812271,0.892167,0,0.007833,0.002067


{'chirp_mass': {'label': 'chirp_mass',
  'label_index': 0,
  'taxonomy_mode': 'prediction',
  'interval_mode': 'asymmetric',
  'n_bins': 24,
  'coverage': 0.9027666666666668,
  'median_width_phys': 15.623655575492135,
  'q90_width_phys': 21.191378383142364,
  'q95_width_phys': 21.7308572319746,
  'min_coverage_per_bin': 0.8863813229571984},
 'total_mass': {'label': 'total_mass',
  'label_index': 1,
  'taxonomy_mode': 'prediction',
  'interval_mode': 'asymmetric',
  'n_bins': 16,
  'coverage': 0.9010666666666668,
  'median_width_phys': 27.919005509654284,
  'q90_width_phys': 37.56544883505717,
  'q95_width_phys': 38.80599842588231,
  'min_coverage_per_bin': 0.8876523582405935},
 'chi_eff': {'label': 'chi_eff',
  'label_index': 2,
  'taxonomy_mode': 'difficulty',
  'interval_mode': 'symmetric',
  'n_bins': 6,
  'coverage': 0.8984,
  'median_width_phys': 0.5762910812388933,
  'q90_width_phys': 0.8122714828905009,
  'q95_width_phys': 0.8122714828905009,
  'min_coverage_per_bin': 0.89216683

In [77]:
assert set(selected_policy_df["label"]) == set(label_names)
assert len(selected_policy_df) == 3

print("Selected Mondrian policy:", MONDRIAN_POLICY)

Selected Mondrian policy: conservative


## 10. Reconstruct selected per-target Mondrian calibrators

The final conservative selection uses different Mondrian systems for different targets. Therefore, real-event inference is performed target-by-target rather than with one single homogeneous Mondrian run.

In [78]:
from src.conformal.binning import QuantileBinner, BinGrouper
from src.conformal.calibration import ConformalIntervalCalibrator
from src.conformal.difficulty import DifficultyEstimator

In [79]:
from dataclasses import dataclass

@dataclass
class SelectedTargetMondrianSystem:
    label: str
    label_index: int
    taxonomy_mode: str
    interval_mode: str
    n_bins: int
    binner: object
    calibrator: object
    difficulty_model: object | None
    intervals_std: np.ndarray
    bin_indices_cal: np.ndarray
    binning_scores_cal: np.ndarray

In [80]:
def fit_selected_target_mondrian_system(
    *,
    label: str,
    label_index: int,
    taxonomy_mode: str,
    interval_mode: str,
    n_bins: int,
    pred_cal: np.ndarray,
    y_cal: np.ndarray,
    emb_cal: np.ndarray,
    confidence_level: float = 0.90,
    n_neighbors: int = 5,
    apply_jitter: bool = False,
    jitter_variation: float = 1e-10,
    min_samples_per_bin: int = 10,
):
    """
    Fit one selected Mondrian conformal system for one target.

    Everything is fitted using calibration data only.

    Parameters
    ----------
    label_index:
        0 -> chirp_mass
        1 -> total_mass
        2 -> chi_eff

    pred_cal, y_cal:
        Standardized calibration predictions and labels.

    emb_cal:
        Calibration embeddings from M08.

    Returns
    -------
    SelectedTargetMondrianSystem
    """
    pred_cal = np.asarray(pred_cal, dtype=np.float64)
    y_cal = np.asarray(y_cal, dtype=np.float64)
    emb_cal = np.asarray(emb_cal, dtype=np.float64)

    assert pred_cal.shape == y_cal.shape
    assert pred_cal.ndim == 2
    assert emb_cal.ndim == 2
    assert pred_cal.shape[0] == emb_cal.shape[0]

    # Residual definition used by your conformal code:
    # residual = y_cal - pred_cal
    residuals_cal_all = y_cal - pred_cal
    residuals_cal_target = residuals_cal_all[:, [label_index]]

    # ------------------------------------------------------------
    # 1. Build binning scores
    # ------------------------------------------------------------
    if taxonomy_mode == "prediction":
        # Prediction taxonomy:
        # score = model prediction for that target.
        binning_scores_cal = pred_cal[:, [label_index]]
        difficulty_model = None

    elif taxonomy_mode == "difficulty":
        # Difficulty taxonomy:
        # fit kNN in embedding space using calibration residuals.
        #
        # Important:
        # We pass residuals for all labels, because your DifficultyEstimator
        # computes one difficulty score per label.
        difficulty_model = DifficultyEstimator(n_neighbors=n_neighbors)
        difficulty_model.calibrate_estimator(
            cal_embedding=emb_cal,
            cal_residuals=residuals_cal_all,
        )

        difficulty_scores_cal_all = difficulty_model.compute_calibration_difficulty()
        binning_scores_cal = difficulty_scores_cal_all[:, [label_index]]

    else:
        raise ValueError(f"Unknown taxonomy_mode: {taxonomy_mode}")

    # ------------------------------------------------------------
    # 2. Fit quantile bin edges using calibration scores
    # ------------------------------------------------------------
    binner = QuantileBinner(
        n_bins=n_bins,
        apply_jitter=apply_jitter,
        jitter_variation=jitter_variation,
    )

    bin_indices_cal = binner.bin_edges_and_indices(binning_scores_cal)

    # ------------------------------------------------------------
    # 3. Group calibration residuals by bin
    # ------------------------------------------------------------
    grouper = BinGrouper()
    grouped_residuals = grouper.group_by_bin(
        residuals=residuals_cal_target,
        bin_indices=bin_indices_cal,
        n_bins=n_bins,
    )

    # ------------------------------------------------------------
    # 4. Fit conformal offsets
    # ------------------------------------------------------------
    calibrator = ConformalIntervalCalibrator(
        confidence_level=confidence_level,
        interval_mode=interval_mode,
        min_samples_per_bin=min_samples_per_bin,
    )
    calibrator.fit(grouped_residuals)

    intervals_std = calibrator.intervals_

    # Since this is one target only, shape should be:
    # (1, n_bins, 2)
    assert intervals_std.shape == (1, n_bins, 2)

    return SelectedTargetMondrianSystem(
        label=label,
        label_index=label_index,
        taxonomy_mode=taxonomy_mode,
        interval_mode=interval_mode,
        n_bins=n_bins,
        binner=binner,
        calibrator=calibrator,
        difficulty_model=difficulty_model,
        intervals_std=intervals_std,
        bin_indices_cal=bin_indices_cal,
        binning_scores_cal=binning_scores_cal,
    )

In [81]:
CONFIDENCE_LEVEL = 0.90
N_NEIGHBORS = 5

# Usa estos nombres tal como ya los tenías
pred_cal = SPLITS["cal"]["pred"]
y_cal = SPLITS["cal"]["y"]
emb_cal = SPLITS["cal"]["emb"]

selected_mondrian_systems = {}

for label, cfg in selected_configs.items():
    print(f"Fitting {label}: {cfg}")

    system = fit_selected_target_mondrian_system(
        label=label,
        label_index=cfg["label_index"],
        taxonomy_mode=cfg["taxonomy_mode"],
        interval_mode=cfg["interval_mode"],
        n_bins=cfg["n_bins"],
        pred_cal=pred_cal,
        y_cal=y_cal,
        emb_cal=emb_cal,
        confidence_level=CONFIDENCE_LEVEL,
        n_neighbors=N_NEIGHBORS,
        apply_jitter=False,
        min_samples_per_bin=10,
    )

    selected_mondrian_systems[label] = system

print("\nFitted systems:")
for label, system in selected_mondrian_systems.items():
    print(
        label,
        "| taxonomy:", system.taxonomy_mode,
        "| interval:", system.interval_mode,
        "| n_bins:", system.n_bins,
        "| intervals shape:", system.intervals_std.shape,
    )

Fitting chirp_mass: {'label': 'chirp_mass', 'label_index': 0, 'taxonomy_mode': 'prediction', 'interval_mode': 'asymmetric', 'n_bins': 24, 'coverage': 0.9027666666666668, 'median_width_phys': 15.623655575492135, 'q90_width_phys': 21.191378383142364, 'q95_width_phys': 21.7308572319746, 'min_coverage_per_bin': 0.8863813229571984}
Fitting total_mass: {'label': 'total_mass', 'label_index': 1, 'taxonomy_mode': 'prediction', 'interval_mode': 'asymmetric', 'n_bins': 16, 'coverage': 0.9010666666666668, 'median_width_phys': 27.919005509654284, 'q90_width_phys': 37.56544883505717, 'q95_width_phys': 38.80599842588231, 'min_coverage_per_bin': 0.8876523582405935}
Fitting chi_eff: {'label': 'chi_eff', 'label_index': 2, 'taxonomy_mode': 'difficulty', 'interval_mode': 'symmetric', 'n_bins': 6, 'coverage': 0.8984, 'median_width_phys': 0.5762910812388933, 'q90_width_phys': 0.8122714828905009, 'q95_width_phys': 0.8122714828905009, 'min_coverage_per_bin': 0.8921668362156663}

Fitted systems:
chirp_mass | t

## 11. Apply calibrators to new predictions

In [82]:
def apply_selected_mondrian_systems(
    *,
    pred_std: np.ndarray,
    emb: np.ndarray,
    selected_mondrian_systems: dict,
):
    """
    Apply selected per-target Mondrian systems to new predictions.

    Parameters
    ----------
    pred_std:
        Standardized M08 predictions, shape (N, 3).

    emb:
        M08 embeddings, shape (N, 64).

    Returns
    -------
    lower_std, upper_std:
        Standardized conformal interval bounds, shape (N, 3).

    bin_indices:
        Assigned bin per sample/target, shape (N, 3).

    binning_scores:
        Binning score per sample/target, shape (N, 3).
    """
    pred_std = np.asarray(pred_std, dtype=np.float64)
    emb = np.asarray(emb, dtype=np.float64)

    if pred_std.ndim != 2 or pred_std.shape[1] != 3:
        raise ValueError(f"Expected pred_std shape (N, 3), got {pred_std.shape}")

    if emb.ndim != 2:
        raise ValueError(f"Expected emb shape (N, embedding_dim), got {emb.shape}")

    if emb.shape[0] != pred_std.shape[0]:
        raise ValueError("pred_std and emb must have the same number of samples.")

    n_samples = pred_std.shape[0]

    lower_std = np.empty_like(pred_std)
    upper_std = np.empty_like(pred_std)
    bin_indices = np.empty_like(pred_std, dtype=int)
    binning_scores = np.empty_like(pred_std)

    for label, system in selected_mondrian_systems.items():
        j = system.label_index

        # ------------------------------------------------------------
        # 1. Compute target binning score
        # ------------------------------------------------------------
        if system.taxonomy_mode == "prediction":
            scores_new = pred_std[:, [j]]

        elif system.taxonomy_mode == "difficulty":
            if system.difficulty_model is None:
                raise ValueError(f"{label} requires a fitted difficulty model.")

            difficulty_scores_all = system.difficulty_model.compute_target_difficulty(
                target_embedding=emb
            )
            scores_new = difficulty_scores_all[:, [j]]

        else:
            raise ValueError(f"Unknown taxonomy_mode: {system.taxonomy_mode}")

        # ------------------------------------------------------------
        # 2. Assign bin
        # ------------------------------------------------------------
        bins_new = system.binner.get_bin_indices(scores_new)

        # Shape checks: one target only
        assert bins_new.shape == (n_samples, 1)

        # ------------------------------------------------------------
        # 3. Select calibrated offsets for assigned bin
        # ------------------------------------------------------------
        lower_offsets = system.intervals_std[0, bins_new[:, 0], 0]
        upper_offsets = system.intervals_std[0, bins_new[:, 0], 1]

        lower_std[:, j] = pred_std[:, j] + lower_offsets
        upper_std[:, j] = pred_std[:, j] + upper_offsets

        bin_indices[:, j] = bins_new[:, 0]
        binning_scores[:, j] = scores_new[:, 0]

    if np.any(lower_std > upper_std):
        raise ValueError("Some intervals have lower > upper.")

    return lower_std, upper_std, bin_indices, binning_scores

In [83]:
def intervals_std_to_phys(lower_std, upper_std):
    """
    Convert standardized interval bounds to physical units.
    """
    lower_std = np.asarray(lower_std, dtype=np.float64)
    upper_std = np.asarray(upper_std, dtype=np.float64)

    lower_phys = inverse_standardize(lower_std)
    upper_phys = inverse_standardize(upper_std)

    return lower_phys, upper_phys

## 12. Validation on synthetic test data

In [84]:
pred_test = SPLITS["test"]["pred"]
y_test = SPLITS["test"]["y"]
emb_test = SPLITS["test"]["emb"]

lower_test_std, upper_test_std, bins_test, scores_test = apply_selected_mondrian_systems(
    pred_std=pred_test,
    emb=emb_test,
    selected_mondrian_systems=selected_mondrian_systems,
)

print("lower_test_std:", lower_test_std.shape)
print("upper_test_std:", upper_test_std.shape)
print("bins_test:", bins_test.shape)

assert lower_test_std.shape == pred_test.shape
assert upper_test_std.shape == pred_test.shape
assert bins_test.shape == pred_test.shape

lower_test_std: (30000, 3)
upper_test_std: (30000, 3)
bins_test: (30000, 3)


In [85]:
covered = (lower_test_std <= y_test) & (y_test <= upper_test_std)
coverage = covered.mean(axis=0)

width_std = upper_test_std - lower_test_std
median_width_std = np.median(width_std, axis=0)

print("Global coverage:")
for j, name in enumerate(label_names):
    print(f"  {name:12s}: {coverage[j]:.6f}")

print("\nMedian width [standardized]:")
for j, name in enumerate(label_names):
    print(f"  {name:12s}: {median_width_std[j]:.6f}")

Global coverage:
  chirp_mass  : 0.899233
  total_mass  : 0.903033
  chi_eff     : 0.903300

Median width [standardized]:
  chirp_mass  : 0.924717
  total_mass  : 0.794647
  chi_eff     : 1.277579


In [86]:
lower_test_phys, upper_test_phys = intervals_std_to_phys(
    lower_test_std,
    upper_test_std,
)

y_test_phys = inverse_standardize(y_test)
pred_test_phys = inverse_standardize(pred_test)

width_phys = upper_test_phys - lower_test_phys
median_width_phys = np.median(width_phys, axis=0)
q90_width_phys = np.quantile(width_phys, 0.90, axis=0)
q95_width_phys = np.quantile(width_phys, 0.95, axis=0)

print("Physical width summary:")
for j, name in enumerate(label_names):
    print(
        f"{name:12s} | "
        f"median={median_width_phys[j]:.6f} | "
        f"q90={q90_width_phys[j]:.6f} | "
        f"q95={q95_width_phys[j]:.6f}"
    )

Physical width summary:
chirp_mass   | median=15.243710 | q90=21.185392 | q95=21.612253
total_mass   | median=27.534664 | q90=38.627888 | q95=38.888468
chi_eff      | median=0.563227 | q90=0.792811 | q95=0.792811


In [87]:
validation_rows = []

for j, name in enumerate(label_names):
    validation_rows.append({
        "label": name,
        "coverage_reconstructed": coverage[j],
        "median_width_phys_reconstructed": median_width_phys[j],
        "q90_width_phys_reconstructed": q90_width_phys[j],
        "q95_width_phys_reconstructed": q95_width_phys[j],
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

display(selected_policy_df[
    [
        "label",
        "coverage",
        "median_width_phys",
        "q90_width_phys",
        "q95_width_phys",
        "min_coverage_per_bin",
    ]
])

,label,coverage_reconstructed,median_width_phys_reconstructed,q90_width_phys_reconstructed,q95_width_phys_reconstructed
0,chirp_mass,0.899233,15.243710,21.185392,21.612253
1,total_mass,0.903033,27.534664,38.627888,38.888468
2,chi_eff,0.903300,0.563227,0.792811,0.792811


,label,coverage,median_width_phys,q90_width_phys,q95_width_phys,min_coverage_per_bin
0,chirp_mass,0.902767,15.623656,21.191378,21.730857,0.886381
1,total_mass,0.901067,27.919006,37.565449,38.805998,0.887652
2,chi_eff,0.898400,0.576291,0.812271,0.812271,0.892167


In [88]:
from src.conformal.pipeline import run_mondrian_regression

def rerun_original_style_for_selected_row(cfg):
    label = cfg["label"]
    j = cfg["label_index"]

    kwargs = dict(
        pred_cal=SPLITS["cal"]["pred"],
        pred_test=SPLITS["test"]["pred"],
        y_cal=SPLITS["cal"]["y"],
        y_test=SPLITS["test"]["y"],
        n_bins=cfg["n_bins"],
        confidence_level=0.90,
        apply_jitter=False,
        interval_mode=cfg["interval_mode"],
        taxonomy_mode=cfg["taxonomy_mode"],
        min_samples_per_bin=10,
        tolerance_sigmas=(1, 2, 3),
    )

    if cfg["taxonomy_mode"] == "difficulty":
        kwargs["cal_embedding"] = SPLITS["cal"]["emb"]
        kwargs["target_embedding"] = SPLITS["test"]["emb"]
        kwargs["n_neighbors"] = 5

    result = run_mondrian_regression(**kwargs)

    lower_phys = inverse_standardize(result.lower)
    upper_phys = inverse_standardize(result.upper)

    widths_phys = upper_phys - lower_phys

    return {
        "label": label,
        "coverage_original_style": result.metrics["global_coverage"][j],
        "median_width_phys_original_style": np.median(widths_phys[:, j]),
        "q90_width_phys_original_style": np.quantile(widths_phys[:, j], 0.90),
        "q95_width_phys_original_style": np.quantile(widths_phys[:, j], 0.95),
    }


original_style_rows = []

for label, cfg in selected_configs.items():
    original_style_rows.append(rerun_original_style_for_selected_row(cfg))

original_style_df = pd.DataFrame(original_style_rows)
display(original_style_df)

display(validation_df)
display(selected_policy_df[
    [
        "label",
        "coverage",
        "median_width_phys",
        "q90_width_phys",
        "q95_width_phys",
    ]
])

,label,coverage_original_style,median_width_phys_original_style,q90_width_phys_original_style,q95_width_phys_original_style
0,chirp_mass,0.899233,15.243710,21.185392,21.612253
1,total_mass,0.903033,27.534664,38.627888,38.888468
2,chi_eff,0.903300,0.563227,0.792811,0.792811


,label,coverage_reconstructed,median_width_phys_reconstructed,q90_width_phys_reconstructed,q95_width_phys_reconstructed
0,chirp_mass,0.899233,15.243710,21.185392,21.612253
1,total_mass,0.903033,27.534664,38.627888,38.888468
2,chi_eff,0.903300,0.563227,0.792811,0.792811


,label,coverage,median_width_phys,q90_width_phys,q95_width_phys
0,chirp_mass,0.902767,15.623656,21.191378,21.730857
1,total_mass,0.901067,27.919006,37.565449,38.805998
2,chi_eff,0.898400,0.576291,0.812271,0.812271


In [89]:
comparison_df = (
    validation_df
    .merge(
        selected_policy_df[
            ["label", "coverage", "median_width_phys", "q90_width_phys", "q95_width_phys"]
        ],
        on="label",
        how="left"
    )
)

comparison_df["delta_coverage"] = (
    comparison_df["coverage_reconstructed"] - comparison_df["coverage"]
)

comparison_df["delta_median_width_phys"] = (
    comparison_df["median_width_phys_reconstructed"] - comparison_df["median_width_phys"]
)

comparison_df["rel_delta_median_width"] = (
    comparison_df["delta_median_width_phys"] / comparison_df["median_width_phys"]
)

display(comparison_df)

,label,coverage_reconstructed,median_width_phys_reconstructed,q90_width_phys_reconstructed,q95_width_phys_reconstructed,coverage,median_width_phys,q90_width_phys,q95_width_phys,delta_coverage,delta_median_width_phys,rel_delta_median_width
0,chirp_mass,0.899233,15.243710,21.185392,21.612253,0.902767,15.623656,21.191378,21.730857,-0.003533,-0.379945,-0.024319
1,total_mass,0.903033,27.534664,38.627888,38.888468,0.901067,27.919006,37.565449,38.805998,0.001967,-0.384342,-0.013766
2,chi_eff,0.903300,0.563227,0.792811,0.792811,0.898400,0.576291,0.812271,0.812271,0.004900,-0.013064,-0.022668


The reconstructed metrics are recomputed from the calibration/test arrays loaded in this notebook. The selected metrics are read from the previous Mondrian summary CSV and are used only to identify the selected hyperparameter configuration. For real-event inference, the reconstructed calibrators are the operative objects, because they are fitted in the current notebook from the loaded calibration arrays.

Al aplicar sobre eventos reales, debo usar los calibradores reconstruidos, no los numeros selected (eso solamente para elegir la mejor config)